 TASK 1: Create Delta Table
 Store data as Delta format

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()

data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)
df.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("/Volumes/workspace/default/outputs/delta")

TASK 2: Insert New Data
(5, "C005", "Camera", 30000)

In [0]:
new_data = [(5, "C005", "Camera", 30000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta").mode("append").save("/Volumes/workspace/default/outputs/delta")

TASK 3: Update Data
Update: id = 2 → amount = 18000


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/workspace/default/outputs/delta")

delta_table.update(
    condition="id = 2",
    set={"amount": "18000"}
)

DataFrame[num_affected_rows: bigint]

TASK 4: Delete Data.
Delete:
id = 1


In [0]:
delta_table.delete("id = 1")
delta_table.toDF().display()

id,customer_id,product,amount
2,C002,Mobile,18000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


TASK 5: MERGE (Incremental Load)
 New incoming data:
updates = [
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)     # insert
]
 Perform UPSERT using MERGE

In [0]:
updates = [
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)     # insert
]

updates_df = spark.createDataFrame(updates, columns)

delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

 TASK 6: Schema Evolution
 Add new column:
country

In [0]:
from pyspark.sql.functions import lit

df_updated = spark.read.format("delta").load("/Volumes/workspace/default/outputs/delta")

df_with_country = df_updated.withColumn("country", lit("India"))

df_with_country.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/outputs/delta")

TASK 7: Time Travel
 Show:
Previous version
Restore old data

In [0]:
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/outputs/delta") \
    .show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
+---+-----------+-------+------+



In [0]:
delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-04-14T16:20:57.000Z,71291458538979,nallagangulaanusha3@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(1330245651905652),8f9713cb-a6a0-415a-a9d1-f64608734791,0414-161009-d3rdnjsg-v2n,7,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1353, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1572)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-14T16:19:04.000Z,71291458538979,nallagangulaanusha3@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1330245651905652),9a5e28cc-d831-4240-afb8-27581744787c,0414-161009-d3rdnjsg-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3807, p25FileSize -> 1353, numDeletionVectorsRemoved -> 1, minFileSize -> 1353, numAddedFiles -> 1, maxFileSize -> 1353, p75FileSize -> 1353, p50FileSize -> 1353, numAddedBytes -> 1353)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-14T16:19:02.000Z,71291458538979,nallagangulaanusha3@gmail.com,MERGE,"Map(predicate -> [""(id#11861L = id#11910L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1330245651905652),9a5e28cc-d831-4240-afb8-27581744787c,0414-161009-d3rdnjsg-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2475, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5308, materializeSourceTimeMs -> 441, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2237, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2442)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-04-14T16:17:53.000Z,71291458538979,nallagangulaanusha3@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1330245651905652),787c27b5-4da8-490b-b821-9ba5dac3779c,0414-161009-d3rdnjsg-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1374, p25FileSize -> 1332, numDeletionVectorsRemoved -> 1, minFileSize -> 1332, numAddedFiles -> 1, maxFileSize -> 1332, p75FileSize -> 1332, p50FileSize -> 1332, numAddedBytes -> 1332)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-14T16:17:51.000Z,71291458538979,nallagangulaanusha3@gmail.com,DELETE,"Map(predicate -> [""(id#11598L = 1)""])",null,List(1330245651905652),787c27b5-4da8-490b-b821-9ba5dac3779c,0414-161009-d3rdnjsg-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2130, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1524, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 605)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-14T16:16:30.000Z,71291458538979,nallagangulaanusha3@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1330245651905652),cdb4f458-1d46-4382-8da1-e35b46cd0665,0414-161009-d3rdnjsg-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3828, p25FileSize -> 1374, numDeletionVectorsRemoved -> 1, minFileSize -> 1374, numAddedFiles -> 1, maxFileSize -> 1374,

In [0]:
delta_table.restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]